In [1]:
!nvidia-smi

Thu Jan 22 07:04:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   44C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import os
PROJECT = "/content/drive/MyDrive/zaawansowane_programowanie/automatic_plate_number_recognition"

os.chdir(PROJECT)
print("cwd:", os.getcwd())
!ls -la

Mounted at /content/drive
cwd: /content/drive/MyDrive/zaawansowane_programowanie/automatic_plate_number_recognition
total 31785
drwx------ 2 root root     4096 Jan 17 22:03  data
drwx------ 2 root root     4096 Jan 18 08:44  models
-rw------- 1 root root        0 Jan 17 20:05 ' README.md'
-rw------- 1 root root      163 Jan 18 11:58  requirements.txt
drwx------ 2 root root     4096 Jan 17 22:02  runs
drwx------ 2 root root     4096 Jan 18 10:56  scripts
drwx------ 2 root root     4096 Jan 17 23:36  src
drwx------ 2 root root     4096 Jan 17 23:05  .venv
drwx------ 2 root root     4096 Jan 17 22:28  .vscode
-rw------- 1 root root  5544453 Jan 18 11:17  yolo26n.pt
-rw------- 1 root root 20422725 Jan 21 21:18  yolo26s.pt
-rw------- 1 root root  6549796 Jan 17 20:50  yolov8n.pt


In [3]:
!pip -q install -U ultralytics==8.4.5 opencv-python-headless

# opcjonalnie (jeśli masz i potrzebujesz)
!pip -q install -r requirements.txt || true


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 MB 41.0 MB/s eta 0:00:00
ERROR: Could not find a version that satisfies the requirement yolo-v8>=0.1.9 (from versions: none)
ERROR: No matching distribution found for yolo-v8>=0.1.9


In [4]:
# zrob link: data/prepared/labels -> data/prepared/labels_yolo
!rm -rf data/prepared/labels
!ln -s labels_yolo data/prepared/labels

!echo "=== data/prepared ==="
!ls -la data/prepared | egrep "labels$|labels_yolo|images"

!echo "=== przykładowe pliki ==="
!ls -la data/prepared/images/train | head
!ls -la data/prepared/labels/train | head

=== data/prepared ===
drwx------ 2 root root  4096 Jan 17 21:40 images
lrw------- 1 root root    11 Jan 22 07:04 labels -> labels_yolo
drwx------ 2 root root  4096 Jan 17 23:12 labels_yolo
=== przykładowe pliki ===
total 310443
-rw------- 1 root root 1415426 Jan 17 20:53 102.jpg
-rw------- 1 root root 1459149 Jan 17 20:53 104.jpg
-rw------- 1 root root 1385885 Jan 17 20:53 105.jpg
-rw------- 1 root root 1361695 Jan 17 20:53 106.jpg
-rw------- 1 root root 1389500 Jan 17 20:53 107.jpg
-rw------- 1 root root 1504095 Jan 17 20:53 109.jpg
-rw------- 1 root root 4860307 Jan 17 20:53 10.jpg
-rw------- 1 root root 1359292 Jan 17 20:53 114.jpg
-rw------- 1 root root 1521378 Jan 17 20:53 115.jpg
total 62
-rw------- 1 root root 38 Jan 17 21:40 102.txt
-rw------- 1 root root 38 Jan 17 21:40 104.txt
-rw------- 1 root root 38 Jan 17 21:40 105.txt
-rw------- 1 root root 38 Jan 17 21:40 106.txt
-rw------- 1 root root 38 Jan 17 21:40 107.txt
-rw------- 1 root root 38 Jan 17 21:40 109.txt
-rw------- 1 r

In [5]:
import glob, os

train_imgs = len(glob.glob("data/prepared/images/train/*.jpg"))
train_lbls = len(glob.glob("data/prepared/labels/train/*.txt"))
val_imgs   = len(glob.glob("data/prepared/images/val/*.jpg"))
val_lbls   = len(glob.glob("data/prepared/labels/val/*.txt"))

print("train imgs:", train_imgs, "train lbls:", train_lbls)
print("val imgs:", val_imgs, "val lbls:", val_lbls)

assert os.path.exists("data/data.yaml"), "Brak data/data.yaml"
assert train_imgs > 0, "Brak obrazów train"
assert train_lbls > 0, "Brak etykiet train (labels)"
assert val_imgs > 0, "Brak obrazów val"
assert val_lbls > 0, "Brak etykiet val (labels)"

print("OK ✅ Dataset wygląda poprawnie.")

train imgs: 123 train lbls: 123
val imgs: 14 val lbls: 14
OK ✅ Dataset wygląda poprawnie.


In [6]:
from ultralytics import YOLO

DATA_YAML = "data/data.yaml"

model = YOLO("yolo26s.pt")

results = model.train(
    data=DATA_YAML,
    epochs=400,
    imgsz=1280,
    device=0,
    batch=-1,
    workers=2,
    cache=False,
    amp=True,
    patience=0,
    project="runs/plate_det",
    name="yolo26s_400e_l4",

    # kluczowe dla małych obiektów / małego datasetu:
    close_mosaic=15,   # ostatnie 15 epok bez mozaiki
    fliplr=0.0,        # nie odwracaj tablic w poziomie
    mixup=0.1,         # delikatnie
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
New https://pypi.org/project/ultralytics/8.4.7 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.5 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA L4, 22693MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=400, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freez

In [7]:
RUN_DIR = "runs/plate_det/yolo26s_400e_l4"

!ls -la {RUN_DIR}/weights

!cp {RUN_DIR}/weights/best.pt /content/drive/MyDrive/best_plate_l4.pt
!cp {RUN_DIR}/weights/last.pt /content/drive/MyDrive/last_plate_l4.pt

!echo "Skopiowano na Drive:"
!ls -la /content/drive/MyDrive | egrep "best_plate_l4|last_plate_l4"

ls: cannot access 'runs/plate_det/yolo26s_400e_l4/weights': No such file or directory
cp: cannot stat 'runs/plate_det/yolo26s_400e_l4/weights/best.pt': No such file or directory
cp: cannot stat 'runs/plate_det/yolo26s_400e_l4/weights/last.pt': No such file or directory
Skopiowano na Drive:
-rw------- 1 root root 20463173 Jan 21 22:16 best_plate_l4.pt
-rw------- 1 root root 20463173 Jan 21 22:16 last_plate_l4.pt


In [8]:
import glob, os

best_candidates = glob.glob("runs/**/best.pt", recursive=True)
last_candidates = glob.glob("runs/**/last.pt", recursive=True)

assert best_candidates, "Nie znalazłem żadnego best.pt w runs/**"
best_path = max(best_candidates, key=os.path.getmtime)

last_path = None
if last_candidates:
    last_path = max(last_candidates, key=os.path.getmtime)

print("BEST:", best_path)
print("LAST:", last_path)

!cp -v "{best_path}" "/content/drive/MyDrive/best_plate_l4.pt"
if last_path:
    !cp -v "{last_path}" "/content/drive/MyDrive/last_plate_l4.pt"

!ls -la /content/drive/MyDrive | egrep "best_plate_l4|last_plate_l4"

BEST: runs/detect/runs/plate_det/yolo26s_400e_l42/weights/best.pt
LAST: runs/detect/runs/plate_det/yolo26s_400e_l42/weights/last.pt
'runs/detect/runs/plate_det/yolo26s_400e_l42/weights/best.pt' -> '/content/drive/MyDrive/best_plate_l4.pt'
'runs/detect/runs/plate_det/yolo26s_400e_l42/weights/last.pt' -> '/content/drive/MyDrive/last_plate_l4.pt'
-rw------- 1 root root 20463173 Jan 22 08:02 best_plate_l4.pt
-rw------- 1 root root 20463173 Jan 22 08:02 last_plate_l4.pt


In [9]:
from ultralytics import YOLO

m = YOLO("runs/plate_det/yolo26s_400e_l4/weights/best.pt")
# podmień na realny obraz z Twojego zbioru:
test_img = "data/prepared/images/val/111.jpg"

res = m.predict(source=test_img, imgsz=640, conf=0.25, device=0, max_det=1, verbose=False)
print(res[0].boxes)

FileNotFoundError: [Errno 2] No such file or directory: 'runs/plate_det/yolo26s_400e_l4/weights/best.pt'

In [ ]:
import glob, os
from ultralytics import YOLO

# znajdź najnowszy best.pt po czasie modyfikacji
best_candidates = glob.glob("runs/**/best.pt", recursive=True)
assert best_candidates, "Nie znaleziono żadnego best.pt w runs/** — trening mógł się nie zapisać albo runy są gdzie indziej."

best_path = max(best_candidates, key=os.path.getmtime)
print("Używam best.pt:", best_path)

m = YOLO(best_path)

test_img = "data/prepared/images/val/111.jpg"
res = m.predict(source=test_img, imgsz=640, conf=0.25, device=0, max_det=1, verbose=False)
print(res[0].boxes)